# svemnet quickstart

Self-Validated Ensemble Models (SVEM) for small-sample design of experiments.

```
pip install "svemnet[formula]"
```

In [ ]:
import numpy as np
import pandas as pd
import svemnet

# Simulate a small designed experiment: 3 continuous factors, n = 24 runs
rng = np.random.default_rng(1)
n = 24
df = pd.DataFrame({
    "Time": rng.uniform(30, 90, n),
    "Temp": rng.uniform(140, 180, n),
    "Conc": rng.uniform(0.5, 2.5, n),
})
df["Yield"] = (
    60
    + 0.20 * (df.Time - 60)
    + 0.45 * (df.Temp - 160)
    - 4.0 * (df.Conc - 1.5) ** 2
    + 0.012 * (df.Time - 60) * (df.Temp - 160)
    + rng.normal(0, 1.0, n)
)
df.head()

## Fit SVEM (lasso / elastic-net base learners)

The formula syntax matches R: `(A + B + C)**2` expands to main effects and
two-way interactions, and `I(A**2)` is a literal square. The helper
`response_surface_formula` builds the standard full-quadratic expansion.

In [ ]:
formula = svemnet.response_surface_formula(
    "Yield", continuous=["Time", "Temp", "Conc"]
)
print(formula)

model = svemnet.svem(formula, df, nBoot=200, seed=1)
model.coef_table().round(3)

## Predictions with bootstrap uncertainty

In [ ]:
grid = pd.DataFrame({
    "Time": np.linspace(30, 90, 25),
    "Temp": 165.0,
    "Conc": 1.5,
})
out = model.predict(grid, interval=True, level=0.90)
pd.DataFrame({"Time": grid.Time, "fit": out["fit"],
              "lwr": out["lwr"], "upr": out["upr"]}).head()

## SVEM with forward-selection base learners

`method="forward"` replaces the lasso path with greedy forward selection per
bootstrap; `selection_frequencies` shows how often each term was in the
winning model — a variable-importance readout.

In [ ]:
fwd = svemnet.svem(formula, df, method="forward", nBoot=200, seed=1)
dict(sorted(fwd.selection_frequencies.items(), key=lambda kv: -kv[1]))

## Deterministic benchmark: forward selection by AICc

In [ ]:
bench = svemnet.forward_aicc(formula, df)
print("selected:", bench.selected_terms)
bench.coef_table().round(3)

## scikit-learn interface

`SVEMRegressor` is a conformant sklearn estimator: use it in pipelines and
cross-validation, with categoricals handled via `ColumnTransformer`.

In [ ]:
from svemnet import SVEMRegressor

X = df[["Time", "Temp", "Conc"]].to_numpy()
y = df["Yield"].to_numpy()
est = SVEMRegressor(method="forward", n_boot=100, random_state=0).fit(X, y)
y_hat, intervals = est.predict_interval(X, confidence_level=0.9)
est.score(X, y)

## Learn more

- README: coming-from-R/JMP mapping and scope notes
- R package (full workflows: binomial responses, whole-model testing,
  optimization): https://CRAN.R-project.org/package=SVEMnet
- Method: Lemkus et al. (2021), doi:10.1016/j.chemolab.2021.104439;
  software paper: Karl (2026), doi:10.1016/j.chemolab.2026.105660